# How to Load Musical Themes Aligned to Recordings (MTD)

The Musical Theme Dataset (MTD; Zalkow, Balke, Arifi-Müller and Müller,
TISMIR 2020, [doi:10.5334/tismir.68](https://doi.org/10.5334/tismir.68))
takes 2,067 themes from Barlow and Morgenstern's *A Dictionary of Musical
Themes*. Each theme is sixteen files: its metadata, an excerpt of a
commercial recording of its movement, an alignment of the excerpt with an
encoding of the theme, and four encodings of the theme as MIDI files, note
lists, PDFs, a Sibelius file and a MusicXML file. This guide reads one
movement with `MtdLoader`:

1. the CD track, a seconds timeline holding each theme's excerpt as a
   child;
2. per theme, one timeline per file, each naming its file and what it was
   made from;
3. per theme, four {{< glossary TimelineGroup >}}s, one per encoding;
4. MTD's alignment as {{< glossary MatchClaim >}}s;

and writes the {{< glossary AlignmentBundle >}} as a TimeToShare manifest.

## Setup

Install the library with `pip install timetoalign`. The example data is a
sample of four themes in the layout of the MTD archive, `MTD_sample.zip`,
fetched by the test-data helper; the full `MTD_1.0.0.zip` is read the same
way, in place, without extracting it. The files that MTD's licence does not
cover (the excerpts, the EDM encodings, the MusicXML files) are stand-ins
shaped like MTD's own; an excerpt is silence as long as MTD's.

Three helpers show what the loader made: `files` one row per timeline of a
theme, `members` one row per member of a theme's groups, and `claims` one
row per MatchClaim from a timeline.

In [1]:
import tempfile
from pathlib import Path

import pandas as pd
from timetoshare import loads, validate

from timetoalign import AlignmentBundle, MatchClaim, MtdLoader
from timetoalign.testdata import ensure_data

ARCHIVE = ensure_data("mtd") / "MTD_sample.zip"


def label(timeline, theme):
    """A timeline's name without its theme: ``SCORE MIDI``, the excerpt ``excerpt``."""
    if timeline.name == f"Theme {theme}":
        return "excerpt"
    return timeline.name.removeprefix(f"{theme}, ")


def theme_timelines(bundle, theme):
    """The excerpt and the file timelines of *theme*, as the bundle holds them."""
    timelines = [bundle.get_timeline(uid) for uid in bundle.timeline_ids]
    children = {uid for timeline in timelines for uid in timeline.list_children()}
    excerpt = next(t for t in timelines if t.name == f"Theme {theme}")
    roots = [
        t for t in timelines if t.id not in children and t.name.startswith(f"{theme}, ")
    ]
    return [excerpt, *roots]


def files(bundle, theme):
    """One row per timeline of *theme*: its axis, length, file and source."""
    timelines = theme_timelines(bundle, theme)
    labels = {f"#{t.id}": label(t, theme) for t in timelines}
    rows = []
    for timeline in timelines:
        statements = timeline.statements
        (item,) = statements["lrmoo:R7i_is_exemplified_by"]
        sources = statements.get("dcterms:source", [])
        rows.append(
            {
                "timeline": label(timeline, theme),
                "unit": timeline.unit.value,
                "length": (
                    "unknown" if timeline.length is None else str(timeline.length)
                ),
                "notes": len(timeline.get_events()),
                "file": item["label"],
                "derived from": ", ".join(labels[s["id"]] for s in sources),
                "children": ", ".join(
                    label(timeline.get_child(uid), theme)
                    for uid in timeline.list_children()
                ),
            }
        )
    return pd.DataFrame(rows)


def members(bundle, theme):
    """One row per member of *theme*'s groups: the group, the member, its length."""
    rows = []
    for group_id in bundle.group_ids:
        group = bundle.get_group(group_id)
        if not group.name.startswith(f"{theme}, "):
            continue
        for uid in group.timeline_ids:
            member = group.get_timeline(uid)
            rows.append(
                {
                    "group": group.name.removeprefix(f"{theme}, "),
                    "member": label(member, theme),
                    "length": (
                        "unknown" if member.length is None else str(member.length)
                    ),
                }
            )
    return pd.DataFrame(rows)


def claims(field, timeline):
    """One row per MatchClaim from *timeline*: the note, the excerpt's side."""
    events = timeline.get_events().table
    pitches = dict(
        zip(events.column("id").to_pylist(), events.column("pitch").to_pylist())
    )
    return pd.DataFrame(
        [
            {
                "onset (ticks)": claim.start_anchor.coordinate_a.value,
                "MIDI pitch": pitches[claim.event_a_id],
                "excerpt side": claim.event_b_id,
                "excerpt (seconds)": claim.start_anchor.coordinate_b.value,
                "claim type": claim.claim_type.value,
            }
            for claim in field
            if claim.timeline_a_id == timeline.id
        ]
    )

## Load one movement

A movement is named as MTD names it, `<ComposerID>_<WorkID>`; the first
movement of Beethoven's Piano Sonata Op. 2 No. 1 is
`Beethoven_Op002No1-01`. The loader reads every theme's metadata to find
the movement's themes, then only their files. Themes can also be named by
their MTD IDs, `themes=["MTD0807"]`; without either, every theme is read.

In [2]:
loader = MtdLoader.from_file(ARCHIVE, movements=["Beethoven_Op002No1-01"])
loader

Sources,1 file(s): MTD_sample.zip
Claims,26
Movements,Beethoven_Op002No1-01
Themes,2
CD tracks,1
Claims matched against,data_EDM-corr_MID
Try,"create_timeline(), create_timelines(), create_bundle()"


Two themes of the movement lie in one CD track of Alfred Brendel's
recording. A track is named as MTD names it: the MusicBrainz release, the
disc and the track.

In [3]:
loader.themes, loader.tracks

(['MTD0807', 'MTD0808'],
 [('814c53fd-0732-4819-9753-4578f5ab992c', '049', 'Track01')])

## The CD track and its excerpts

`create_bundle()` assembles the bundle. Its first timeline is the CD
track, named after its movement, its performer, the disc and the track.
MTD does not state how long the track is, so its length is unknown,
`None`.

In [4]:
bundle = loader.create_bundle()
track = bundle.get_timeline(bundle.timeline_ids[0])
track.name, track.length

('Beethoven Op002No1-01, Brendel (CD 49, track 1)', None)

Each theme's excerpt is a child of the track, placed at the theme's
`StartTime` and as long as `EndTime − StartTime`, both read as the
decimals MTD writes.

In [5]:
excerpts = [track.get_child(uid) for uid in track.list_children()]
pd.DataFrame(
    {
        "excerpt": [excerpt.name for excerpt in excerpts],
        "offset (seconds)": [track.get_child_offset(e.id).value for e in excerpts],
        "length (seconds)": [excerpt.length.value for excerpt in excerpts],
    }
)

,excerpt,offset (seconds),length (seconds)
0,Theme B689 (MTD 0807),0.2,2.8
1,Theme B690 (MTD 0808),84.7,4.5


The track states what MTD says of the recording: that it records the
movement, its release on MusicBrainz, the collection, disc and track as
MTD names them, the performer, and that the recording is in copyright and
identified here, not shared.

In [6]:
track.statements

{'dcterms:accessRights': [{'id': 'http://purl.org/coar/access_right/c_14cb'}],
 'dcterms:rights': [{'id': 'http://rightsstatements.org/vocab/InC/1.0/'}],
 'melod:hasIdentifier': [{'type': ['melod:Identifier'],
   'rdfs:label': ['MTD CollectionID'],
   'owl:hasValue': ['Composer/Beethoven_CompleteEdition_BC']},
  {'type': ['melod:Identifier'],
   'rdfs:label': ['MTD NameCD'],
   'owl:hasValue': ['049']},
  {'type': ['melod:Identifier'],
   'rdfs:label': ['MTD NameTrack'],
   'owl:hasValue': ['Track01']}],
 'rel:prf': [{'label': 'Brendel'}],
 'schema:inAlbum': [{'id': 'http://musicbrainz.org/release/814c53fd-0732-4819-9753-4578f5ab992c'}],
 'schema:recordingOf': [{'label': 'Beethoven Op002No1-01'}],
 'type': ['schema:MusicRecording']}

## The files of one theme

The first theme, B689 in Barlow and Morgenstern, is fourteen timelines:
the excerpt and one per file of its four encodings. A MIDI file is a
ticks timeline, on the file's own axis, with a C-Map to quarters (480
ticks per quarter here); the MusicXML file and the Sibelius file are
quarters timelines, a note list (CSV) and the excerpt seconds timelines,
a PDF a points timeline. Each
names its file in the archive (`lrmoo:R7i_is_exemplified_by`) and, where
MTD's paper states that the file was made from another, that file's
timeline (`dcterms:source`): the SCORE exports derive from the Sibelius
file, EDM-corr's MIDI file from EDM-orig's, each EDM encoding's note list
and PDF from its MIDI file, and EDM-alig's files from EDM-corr's MIDI file.

A timeline is as long as its file states: a MIDI file its end of track,
the MusicXML file the end of its last bar, the excerpt its seconds. A note
list, a PDF and the Sibelius file state no end: their length is unknown
(`length` is `None`). Only the MIDI files' timelines hold notes.

In [7]:
theme = excerpts[0].name.removeprefix("Theme ")
files(bundle, theme)

,timeline,unit,length,notes,file,derived from,children
0,excerpt,seconds,2.8 seconds,0,MTD/data_AUDIO/MTD0807_Beethoven_Op002No1-01.wav,,
1,EDM-orig MIDI,ticks,3839 ticks,10,MTD/data_EDM-orig_MID/MTD0807_Beethoven_Op002N...,,
2,EDM-orig CSV,seconds,unknown,0,MTD/data_EDM-orig_CSV/MTD0807_Beethoven_Op002N...,EDM-orig MIDI,
3,EDM-orig PDF,points,unknown,0,MTD/data_EDM-orig_IMG/MTD0807_Beethoven_Op002N...,EDM-orig MIDI,
4,EDM-corr MIDI,ticks,3840 ticks,10,MTD/data_EDM-corr_MID/MTD0807_Beethoven_Op002N...,EDM-orig MIDI,
5,EDM-corr CSV,seconds,unknown,0,MTD/data_EDM-corr_CSV/MTD0807_Beethoven_Op002N...,EDM-corr MIDI,
6,EDM-corr PDF,points,unknown,0,MTD/data_EDM-corr_IMG/MTD0807_Beethoven_Op002N...,EDM-corr MIDI,
7,SCORE Sibelius,quarters,unknown,0,MTD/data_SCORE_SIB/MTD0807_Beethoven_Op002No1-...,,
8,SCORE MusicXML,quarters,9 quarters,0,MTD/data_SCORE_XML/MTD0807_Beethoven_Op002No1-...,SCORE Sibelius,"SCORE MusicXML, span of the SCORE MIDI"
9,SCORE MIDI,ticks,3840 ticks,10,MTD/data_SCORE_MID/MTD0807_Beethoven_Op002No1-...,SCORE Sibelius,


The theme's identifiers and its movement are stated on the two timelines
nothing is derived from, EDM-orig's MIDI file and the Sibelius file.

In [8]:
edm_orig = next(
    t for t in theme_timelines(bundle, theme) if label(t, theme) == "EDM-orig MIDI"
)
edm_orig.statements

{'lrmoo:R7i_is_exemplified_by': [{'type': ['melod:Item'],
   'label': 'MTD/data_EDM-orig_MID/MTD0807_Beethoven_Op002No1-01.mid',
   'schema:isPartOf': [{'id': 'https://www.audiolabs-erlangen.de/resources/MIR/MTD/MTD_1.0.0.zip'}]}],
 'melod:hasIdentifier': [{'type': ['melod:Identifier'],
   'rdfs:label': ['MTD'],
   'owl:hasValue': ['0807']},
  {'type': ['melod:Identifier'],
   'rdfs:label': ['Barlow and Morgenstern'],
   'owl:hasValue': ['B689']}],
 'schema:isPartOf': [{'label': 'Beethoven Op002No1-01'}]}

## Four groups, one per encoding

Each encoding of the theme is a group: EDM-orig, EDM-corr and SCORE of
their files, EDM-alig of its two files and the excerpt, to which
EDM-alig is aligned (its quarter notes are the excerpt's seconds). The
members of a group share its start and end. Two pairs of members state
different lengths: EDM-alig's MIDI file ends at tick 1369, 25 ticks past
the 1344 ticks (2.8 quarters) of the excerpt's 2.8 seconds, since the
aligned notes run past the excerpt's cut; the MusicXML file's last bar
ends at 9 quarters, past the SCORE MIDI file's end of track at tick 3840,
8 quarters. Nothing is stretched to fit: the
longer member takes part through a child of it, starting at 0 and as long
as the shorter one, named after the span it covers (the children of the
table above).

In [9]:
members(bundle, theme)

,group,member,length
0,EDM-orig,EDM-orig MIDI,3839 ticks
1,EDM-orig,EDM-orig CSV,unknown
2,EDM-orig,EDM-orig PDF,unknown
3,EDM-corr,EDM-corr MIDI,3840 ticks
4,EDM-corr,EDM-corr CSV,unknown
5,EDM-corr,EDM-corr PDF,unknown
6,SCORE,SCORE Sibelius,unknown
7,SCORE,"SCORE MusicXML, span of the SCORE MIDI",8 quarters
8,SCORE,SCORE MIDI,3840 ticks
9,SCORE,SCORE CSV,unknown


Every group holds a member of unknown length (a note list, a PDF or the
Sibelius file), so a group converts no position until those lengths are
known.

## The recording's transposition

The notes keep their encoding's pitches. META's `MidiTransposition`, the
semitones by which the recording sounds transposed against the encodings,
is a comment on the excerpt: 0 for this theme. The finale of the Fifth
Symphony, read by its MTD ID, states 5.

In [10]:
excerpts[0].statements["rdfs:comment"]

["MTD's MidiTransposition: 0 semitones, the recording against the symbolic encodings"]

In [11]:
fifth = MtdLoader.from_file(ARCHIVE, themes=["MTD1080"]).create_bundle()
fifth_track = fifth.get_timeline(fifth.timeline_ids[0])
fifth_track.get_child(fifth_track.list_children()[0]).statements["rdfs:comment"]

["MTD's MidiTransposition: 5 semitones, the recording against the symbolic encodings"]

## MTD's alignment as MatchClaims

The alignment file has rows of quarters in EDM-corr's MIDI file and
seconds in the excerpt. The loader reads each row but the last as
synchronous MatchClaims from each note at that onset of the EDM-corr MIDI
timeline to the sentinel `CHUNK` on the excerpt (written `CHUNK_SENTINEL`
in the MatchClaim table): the excerpt holds no event, the claim states a
position on it. The file writes the onset with five decimals (`6.66667`);
the claim holds the onset of the note it meets on the MIDI file's own
axis, tick 3200. The last row, the end of the last note, is checked and
not kept. The loader hands the claims over as one field.

In [12]:
field = loader.get_field(MatchClaim)
edm_corr = next(
    t for t in theme_timelines(bundle, theme) if label(t, theme) == "EDM-corr MIDI"
)
claims(field, edm_corr)

,onset (ticks),MIDI pitch,excerpt side,excerpt (seconds),claim type
0,0,60,CHUNK_SENTINEL,0.12855,event_match
1,480,65,CHUNK_SENTINEL,0.45519,event_match
2,960,68,CHUNK_SENTINEL,0.75268,event_match
3,1440,72,CHUNK_SENTINEL,1.06608,event_match
4,1920,77,CHUNK_SENTINEL,1.38863,event_match
5,2400,80,CHUNK_SENTINEL,1.71300,event_match
6,3120,79,CHUNK_SENTINEL,2.13951,event_match
7,3200,77,CHUNK_SENTINEL,2.20401,event_match
8,3280,76,CHUNK_SENTINEL,2.28850,event_match
9,3360,77,CHUNK_SENTINEL,2.35321,event_match


The timeline's C-Map reads tick 3200 as exactly `20/3` quarters, the
onset the alignment file writes as `6.66667`.

In [13]:
edm_corr.get_timestamp_at(3200)

ID,Coordinate,Type
dlt_0bfv362erabkd,3200 ticks,axis
quarters,20/3 quarters,cmap


The bundle holds the same claims (`bundle.match_claim_table`) beside its
groups, whose members of unknown length leave its matchstamps without an
answer. A bundle of the timelines and the claims without the groups
answers one: from the triplet's onset, the excerpt's second and, through
the excerpt's offset, the track's.

In [14]:
plain = AlignmentBundle(name="timelines and claims")
for timeline in loader.create_timelines():
    plain.add_timeline(timeline)
plain.add_match_claim_field(field)
plain.get_matchstamp_at(3200, edm_corr.id)

ID,Coordinate,Type
dlt_0bfv362erabkd,3200 ticks,stated
cpt_21te1bczlwez7,2.40401 seconds,inferred
cpt_3g6ye3gp86n8k,2.20401 seconds,stated


## Track lengths from elsewhere

A track length known from elsewhere, MusicBrainz for instance, is given to
`create_bundle` by the track's key (431 seconds here, for illustration).

In [15]:
(key,) = loader.tracks
loader.create_bundle(track_lengths={key: 431.0}).get_timeline(track.id).length

Coordinate(431.0, seconds)

## Write the manifest

`to_file()` writes the manifest and, beside it, the MatchClaim table
(`match_claims.parquet`), which the manifest names among its resources.
With `events=False` the notes are not written; the claims keep their IDs.

In [16]:
package = Path(tempfile.mkdtemp()) / "beethoven_op002no1-01"
package.mkdir()
manifest = bundle.to_file(package / "manifest.yaml", events=False)
sorted(path.name for path in package.iterdir())

['manifest.yaml', 'match_claims.parquet']

`to_file()` validates the manifest before writing it; TimeToShare's
`validate` repeats the check on the file and raises at the first
violation. `from_file()` reads the manifest and its table back: the
timelines (children included), the groups and the claims.

In [17]:
validate(loads(manifest.read_text(encoding="utf-8")))
read = AlignmentBundle.from_file(manifest)
len(read.timeline_ids), len(read.group_ids), len(read.match_claim_table)

(33, 8, 26)